In [0]:
%pip install databricks-feature-engineering -q

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. Configuración de Unity Catalog y Feature Store
fe = FeatureEngineeringClient()
CATALOG = "workspace"
SCHEMA = "gold"
FEATURE_TABLE = f"{CATALOG}.{SCHEMA}.store_performance_features"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.walmart_sales_predictor"

# 2. Definir los "Lookups" del Feature Store
feature_lookups = [
    FeatureLookup(
      table_name=FEATURE_TABLE,
      lookup_key="store_id",
      timestamp_lookup_key="sales_date",
      feature_names=["feat_rolling_transactions", "feat_rolling_revenue", "feat_items_sold"]
    )
]

# 3. Crear el Training Set
raw_data = spark.table(f"{CATALOG}.gold.regional_sales_performance").select("store_id", "sales_date", "total_revenue")

training_set = fe.create_training_set(
    df=raw_data,
    feature_lookups=feature_lookups,
    label="total_revenue", 
    exclude_columns=["sales_date", "store_id"]
)

training_df = training_set.load_df().toPandas().dropna()
X = training_df.drop(["total_revenue"], axis=1)
y = training_df["total_revenue"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 4. Entrenamiento con MLflow (Tracking)
with mlflow.start_run(run_name="Walmart_Sales_Forecasting") as run:
    # Definir y entrenar el modelo
    params = {"n_estimators": 100, "random_state": 42}
    mlflow.log_params(params)
    
    rf = RandomForestRegressor(**params)
    rf.fit(X_train, y_train)
    
    # --- CÁLCULO DE MÉTRICAS ---
    predictions = rf.predict(X_test)
    rmse = mean_squared_error(y_test, predictions) ** 0.5
    mae = mean_absolute_error(y_test, predictions)
    
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae) # El error promedio en unidades reales
    
    # --- GENERAR IMPORTANCIA DE CARACTERÍSTICAS (FEATURE IMPORTANCE) ---
    importances = rf.feature_importances_
    feature_names = X.columns
    feature_importance_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values(by="importance", ascending=False)
    
    # Crear gráfica
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance_df["feature"], feature_importance_df["importance"], color='#2ecc71')
    plt.xlabel("Importancia")
    plt.title("Walmart Predictor: Factores Influyentes")
    plt.gca().invert_yaxis()
    
    # Guardar y registrar la gráfica como artefacto
    plot_path = "/tmp/feature_importance.png"
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path, artifact_path="plots")
    
    # Registrar tabla de importancia como CSV
    csv_path = "/tmp/feature_importance.csv"
    feature_importance_df.to_csv(csv_path, index=False)
    mlflow.log_artifact(csv_path, artifact_path="metadata")
    
    # 5. Registrar el modelo en Unity Catalog
    fe.log_model(
        model=rf,
        artifact_path="sales_model",
        flavor=mlflow.sklearn,
        training_set=training_set,
        registered_model_name=MODEL_NAME,
        input_example=X_train[:5]
    )
    
    print(f"✅ Éxito. RMSE: {rmse:.2f} | MAE: {mae:.2f}")
    print(f"📍 Modelo registrado: {MODEL_NAME}")